# 🏥 Medical Report Analysis System — Google Colab Runner

This notebook configures and runs the full 3-model medical pipeline (**LLaMA 3 8B → Meditron 7B → BioMistral 7B**) with GPU acceleration, and provides a full framework to **LoRA Fine-Tune** the model on Colab.

### ⚙️ Google Colab Requirements
- **Runtime**: You **MUST** use a GPU runtime (T4, L4, or A100). Go to `Runtime > Change runtime type` and select **GPU**.
- **Kaggle API Key**: Have your `kaggle.json` file ready to upload for the dataset downloads.

## 📥 Step 0: Clone the Repository & Mount Drive
We start by mounting your Google Drive (so downloaded files save permanently) and cloning the latest code from your GitHub repository. This guarantees that even if you only uploaded this notebook, all the Python scripts will automatically download!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/URVISH611711/report-analysis.git
%cd report-analysis

## 🛠️ Step 1: Environment Setup & GPU Installation
We compile `llama-cpp-python` with **CUDA support** to offload model execution to the Colab GPU, making inference extremely fast. We also install HuggingFace training dependencies.

In [ ]:
# 1. Install GPU-accelerated llama-cpp-python
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python==0.3.19 --no-cache-dir

# 2. Install all other project dependencies
!pip install pymupdf pytesseract pillow paddleocr gradio fpdf2 python-dotenv regex datasets kaggle huggingface_hub pandas tqdm transformers peft trl accelerate bitsandbytes

## 📂 Step 2: Upload credentials (`kaggle.json`)
Run the cell below and upload the `kaggle.json` file to enable Kaggle medical dataset downloads.

In [ ]:
from google.colab import files
import os
import shutil

uploaded = files.upload()
if "kaggle.json" in uploaded:
    print("\n[+] Credentials loaded successfully!")
else:
    print("\n[!] Please upload kaggle.json to download the Kaggle datasets.")

## 📥 Step 3: Run Model and Dataset Downloads
Because of our recent script updates, running these Python scripts will automatically detect Google Colab and download the models and datasets straight into your Google Drive permanently!

In [ ]:
# Download the 3 GGUF models (LLaMA 3, Meditron, BioMistral) directly to Google Drive
!python download_models.py

In [ ]:
# Download the 6 training datasets directly to Google Drive
!python download_datasets.py

## 🚀 Step 4: Launch Web Dashboard
Run this cell to start the Gradio web dashboard. It will automatically detect Colab and generate a public **`https://*.gradio.live`** URL.

In [ ]:
# Run Gradio interface
!python app/main.py

## 🧠 Step 5: How to Fine-Tune via QLoRA / LoRA

To customize/fine-tune the BioMistral model on the medical datasets you downloaded, follow these sub-steps:

In [ ]:
# A. Prepare Training Dataset
!python fine_tuning/dataset_prep.py

In [ ]:
# B. Run LoRA Training Session
!python fine_tuning/train_lora.py

In [ ]:
# C. Merge PEFT Adapters Back into Base Model
!python fine_tuning/merge_lora.py